# SI4006 · M5 · Amparo — **Destilar un componente** (S14)

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 5

---

Sigue el lab de la profe (`sesiones/s14/S14_Lab_Destilacion_RESUELTO.ipynb`), **Parte B · datos sintéticos + LoRA**:
el profesor escribe el dataset, el estudiante se fine-tunea con la receta de S04 y se mide con **nuestro harness de M2**.

- **Profesor:** el brazo `4 · agente + reglas` de M3 (gpt-oss-120b por Groq, 3–7 llamadas por caso).
- **Estudiante:** `Qwen2.5-0.5B-Instruct` + LoRA, una sola llamada, sin herramientas, en la misma GPU que el VLM.
- **Qué aprende:** solo el **JSON de hechos**. El mecanismo lo sigue decidiendo el motor del art. 6 (`reglas.decidir`):
  el estudiante no tiene que saber derecho y no puede inventar normas.

La lógica vive en `amparo/pipelines/m5_destilacion.py`; este notebook solo la llama y muestra lo que sale.

> GPU **T4** y `GROQ_API_KEY` en los secretos de Colab. Lo caro es Groq (~5 k tokens por relato que etiqueta el profesor):
> todo se guarda caso a caso y **se retoma donde quedó** si se acaba el cupo diario.

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.45" datasets accelerate peft scikit-learn pandas groq sentence-transformers
print('Listo.')

In [ ]:
# Colab arranca vacío: se clona el repo (idempotente). Si el notebook ya corre dentro del repo, usa esa copia.
import os, sys, subprocess
from pathlib import Path

REPO_URL, RAMA = 'https://github.com/TomasPosada0626/Amparo.git', 'm5'
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'amparo' / 'nucleo' / 'schema.py').exists()), None)
if RAIZ is None:
    if not Path('Amparo').is_dir():
        subprocess.run(['git', 'clone', '-q', '-b', RAMA, REPO_URL, 'Amparo'], check=True)
    RAIZ = Path('Amparo').resolve()
sys.path.insert(0, str(RAIZ))

# Los intermedios caros (relatos, pares, caché del profesor, adaptador) van a Drive para sobrevivir al reinicio.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/amparo_m5'
except ImportError:
    CARPETA = None                      # local: runs/m5/ (ignorada por git)

from amparo.pipelines import m5_destilacion as m5

## 1 · ¿Qué componente? El scorecard de M4

Slide 15, paso 1: *"miren su scorecard de M4: la columna `segundos` dice dónde se va el tiempo"*.
`segundos` es VLM + M3 por caso; las trazas guardan aparte lo que tardó el VLM (`segundos_ver`). La resta es M3.

In [ ]:
try:
    comp = m5.latencia_por_componente()
except FileNotFoundError as ex:      # la corrida de demo de M4 vive en main
    print(ex)

**Lectura.** En la corrida de demo (L4, 11 casos) M3 se lleva ~59 % de la mediana (6.4 s de 10.8 s) y **toda la variación**
(de 3.3 a 14.8 s, según cuántos pasos dé el agente). El VLM tarda ~3.7 s y casi no varía.

Slide 15, paso 2: *¿esa pieza necesita ser grande?* El agente usa un modelo de 117 B para **marcar 13 booleanos y tres frases**:
la decisión jurídica ya la toma el motor de reglas. No necesita ser grande.
Paso 3: es el generador → **forma B**. La A (logits) no aplica: el profesor es una API y no da logits
(slide 12: la B *"sirve con una API cerrada"*).

### El contrato (TODO 1), antes de entrenar

Se copia al README de M5. *"Si no lo escriben antes, después cualquier resultado parece bueno."*

**Temperatura.** En la parte B no hay temperatura de destilación (esa es la T de la KL de la parte A, que necesita logits).
Hay dos temperaturas de **generación**, elegidas para bajar el riesgo:
- el profesor **escribe relatos** a `T = 0.7`, como en el lab → variedad;
- el profesor **responde** a `T = 0` con `seed = 42` (ya es así en `agente.py`) → la etiqueta es reproducible y no se le enseña ruido al estudiante;
- el estudiante responde **codicioso** (`do_sample=False`), como el `preguntar` de S04.

In [ ]:
ctx = m5.preparar(CARPETA)          # raíz, carpeta de trabajo, clave de Groq y el contrato

## 2 · B1 · El profesor escribe los pares

Primero arma el profesor: el brazo 4 real de M3, igual que en M4. Después escribe relatos nuevos
(la celda B1 del lab con `TEMAS` = las 131 categorías del dataset de M1) y los responde.
Los relatos que se parecen demasiado al eval set se descartan (fuga).

In [ ]:
m5.armar_profesor(ctx);

In [ ]:
rel = m5.escribir_relatos(ctx)            # N_POR_TEMA = 2 → ~262 relatos; se retoma si se corta
for r in rel[:3]:
    print(f"[{r['categoria']}] {r['relato']}\n")

In [ ]:
pares = m5.etiquetar_con_profesor(ctx)    # sintéticos + los 160 de train/validation de M1
import json
for p in pares[:3]:
    print(p['relato'][:160], '\n  →', json.dumps(p['objetivo'], ensure_ascii=False), '\n')

### Etiqueta dura vs. etiqueta del profesor

La celda de la profe compara la etiqueta del dataset con la del profesor. Aquí: con los hechos del **profesor**,
¿cuántas veces el motor llega al mecanismo gold en los relatos de M1? Lo que el profesor falla es ruido que el estudiante
va a heredar (slide 19).

In [ ]:
acuerdo = m5.acuerdo_profesor_gold(ctx)

## 3 · B2 · El estudiante: Qwen2.5-0.5B + LoRA (receta de S04)

`r=8`, `lora_alpha=16`, `q_proj`/`v_proj`, `lr=2e-4`, 3 épocas. De M1 se conserva lo que ya se pagó: pérdida solo en la
respuesta, evaluación por época contra validation y el mejor checkpoint. Si el adaptador ya está en la carpeta de trabajo,
se carga en vez de reentrenar.

In [ ]:
if (ctx.carpeta / 'estudiante_lora').exists() and (ctx.carpeta / 'entrenamiento.json').exists():
    m5.cargar_estudiante(ctx)
else:
    m5.entrenar_estudiante(ctx)
print(json.dumps(ctx.entrenamiento.get('seleccion', {}), ensure_ascii=False, indent=1))

In [ ]:
# El estudiante sobre 3 relatos de validation, junto a lo que dijo el profesor
for p in [p for p in ctx.pares if p['split'] != 'train'][:3]:
    print(p['relato'][:140])
    print('  profesor  :', m5.a_texto(p['objetivo']))
    print('  estudiante:', ctx.generar(p['relato']), '\n')

## 4 · B3 · Medir con el harness de M2

`harness(eval_set, sistema)` de M2, para el profesor y para el estudiante, sobre **los mismos 130 casos** con los que se midió
el brazo 4 en M3 (`eval_set_v3` + `eval_set_m3`), **en la misma máquina**, con latencia por caso.
El profesor se mide con la caché vacía: con caché, la latencia mediría la caché.

Primero una prueba rápida con 10 casos; después los 130.

In [ ]:
m5.medir(ctx, n=10)
print(m5.tabla(ctx).T.to_string())

In [ ]:
m5.medir(ctx)                              # los 130 (se retoma si se corta)

## 5 · La tabla antes/después y el contrato

In [ ]:
df = m5.tabla(ctx)
display(df.T)

### Donde el estudiante falla y el profesor no (TODO 4)

Ahí está lo que la destilación no pudo copiar. Fíjense en tres cosas de la slide 19: ¿son adversariales (sesgos heredados)?,
¿categorías raras?, ¿se perdió la válvula de abstención (`acierto_abstencion` en la tabla)?

In [ ]:
perdidos = m5.casos_perdidos(ctx, mostrar=8)

In [ ]:
m5.guardar(ctx)                            # m5/destilacion.json y m5/pares_profesor.json → al repo

## Lo que va a M5

- **La tabla antes/después** de arriba (calidad con el harness de M2, latencia p50/p95, tamaño) y el veredicto del contrato.
- **El contrato** y **las licencias**: profesor gpt-oss-120b (Apache 2.0, permite entrenar con sus salidas) y estudiante Qwen2.5-0.5B (Apache 2.0).
- **Qué cedió el estudiante** (sección *limitaciones y ética*): los casos perdidos, la herramienta de términos (`calcular_termino`,
  que el estudiante no tiene) y la recuperación de normas (el estudiante no consulta el corpus: el fundamento lo pone el motor).
- En S15: el adaptador (`estudiante_lora/`) se cuantiza y se sube con el Space.

*Amparo · M5 · S14.*